# PCA

## PCA from Scratch

ImplementING Principal Component Analysis (PCA) from scratch using only `NumPy`.

* Testing the implementation using small synthetic datasets.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
class PCA:
    """
    Principal Component Analysis implementation using only NumPy.
    """

    def __init__(self, n_components=2):
        """
        Initialize PCA.
        """
        self.n_components = n_components

    def fit(self, X):
        """
        Fit PCA on the training data X.
        """
        # Center the data by subtracting the mean
        self.mean_ = np.mean(X, axis=0)
        X_centered = X - self.mean_

        # Compute unbiased covariance matrix (matches sklearn: divide by n-1)
        n_samples = X.shape[0]
        cov_matrix = np.dot(X_centered.T, X_centered) / (n_samples - 1)

        # Eigen decomposition: eigenvalues and eigenvectors
        eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)

        # Sort eigenvectors by eigenvalues in descending order
        idx = np.argsort(eigenvalues)[::-1]
        eigenvalues = eigenvalues[idx]
        eigenvectors = eigenvectors[:, idx]

        # Select top n_components and transpose to (n_components, n_features)
        self.components_ = eigenvectors[:, :self.n_components].T
        self.explained_variance_ = eigenvalues[:self.n_components]
        self.explained_variance_ratio_ = eigenvalues[:self.n_components] / np.sum(eigenvalues)

    def transform(self, X, dim=None):
        """
        Transform X into the principal component space.
        """
        if dim is None:
            dim = self.n_components

        # Center the data
        X_centered = X - self.mean_

        # Project onto the principal components
        components_to_use = self.components_[:dim].T  # Transpose to (n_features, dim)
        return np.dot(X_centered, components_to_use)

    def inverse_transform(self, X):
        """
        Transform data back to original space.
        """
        # Reconstruct using the top n_components
        X_reconstructed = np.dot(X, self.components_) + self.mean_
        return X_reconstructed


    def fit_transform(self, X):
        """
        Fit PCA and transform X in one step.
        """
        self.fit(X)
        return self.transform(X)


In [ ]:
def test_basic_pca():
    """
    Test 1: Basic PCA on 2D data
    """
    print("Test 1: Basic PCA on 2D data")

    np.random.seed(42)
    X = np.random.randn(50, 2)

    pca = PCA(n_components=2)
    X_transformed = pca.fit_transform(X)

    # Check shape of transformed data
    assert X_transformed.shape == (50, 2), "Transformed shape mismatch"

    # Check reconstruction
    X_reconstructed = pca.inverse_transform(X_transformed)
    reconstruction_error = np.mean((X - X_reconstructed) ** 2)
    assert reconstruction_error < 1e-10, f"Reconstruction error too high: {reconstruction_error}"

    print("✓ Test 1 passed\n")

def test_dimensionality_reduction():
    """
    Test 2: Reduce 5D data to 2D
    """
    print("Test 2: Reduce 5D data to 2D")

    np.random.seed(42)
    X = np.random.randn(100, 5)

    pca = PCA(n_components=2)
    X_transformed = pca.fit_transform(X)

    # Check shape of transformed data
    assert X_transformed.shape == (100, 2), "Transformed shape mismatch"

    print("✓ Test 2 passed\n")

def test_reconstruction():
    """
    Test 3: Inverse transform (reconstruction)
    """
    print("Test 3: Inverse transform (reconstruction)")

    np.random.seed(42)
    X = np.random.randn(50, 3)

    pca = PCA(n_components=2)
    X_transformed = pca.fit_transform(X)
    X_reconstructed = pca.inverse_transform(X_transformed)

    # Check reconstruction error (relaxed for reduced dimensions)
    reconstruction_error = np.mean((X - X_reconstructed) ** 2)
    total_variance = np.var(X)
    relative_error = reconstruction_error / total_variance
    assert reconstruction_error < 0.5, f"Reconstruction error too high: {reconstruction_error} (relative: {relative_error:.4f})"

    print(f"Reconstruction MSE: {reconstruction_error:.6f} (expected ~0.3 for 1 discarded dim)")
    print("✓ Test 3 passed\n")

def test_variance_ordering():
    """Test 4: Components are ordered by variance"""
    print("Test 4: Verify components are ordered by explained variance")

    np.random.seed(42)
    X = np.random.randn(100, 5)

    pca = PCA(n_components=5)
    pca.fit(X)

    # Check that explained variances are in descending order
    variances = pca.explained_variance_
    is_sorted = np.all(variances[:-1] >= variances[1:])

    print(f"Explained variances: {variances}")
    print(f"Is sorted (descending): {is_sorted}")
    assert is_sorted, "Components not sorted by variance!"
    print("✓ Test 4 passed\n")


def test_centered_data():
    """Test 5: Verify data is properly centered"""
    print("Test 5: Verify data centering")

    np.random.seed(42)
    X = np.random.randn(100, 3) + 10  # Add offset

    pca = PCA(n_components=2)
    pca.fit(X)

    # Mean should be close to the original data mean
    print(f"Original data mean: {np.mean(X, axis=0)}")
    print(f"Stored mean: {pca.mean_}")
    print(f"Difference: {np.mean(np.abs(np.mean(X, axis=0) - pca.mean_)):.10f}")
    print("✓ Test 5 passed\n")


def run_all_tests():
    print("Running PCA test suite...\n")
    try:
        test_basic_pca()
        test_dimensionality_reduction()
        test_reconstruction()
        test_variance_ordering()
        test_centered_data()

        print("ALL TESTS PASSED!")

    except AssertionError as e:
        print(f"\n❌ Test failed: {e}")
    except Exception as e:
        print(f"\n❌ Unexpected error: {e}")

In [ ]:
# Run the test suite
run_all_tests()

Running PCA test suite...

Test 1: Basic PCA on 2D data
✓ Test 1 passed

Test 2: Reduce 5D data to 2D
✓ Test 2 passed

Test 3: Inverse transform (reconstruction)
Reconstruction MSE: 0.156774 (expected ~0.3 for 1 discarded dim)
✓ Test 3 passed

Test 4: Verify components are ordered by explained variance
Explained variances: [1.26528496 1.03990633 0.97435577 0.87353896 0.66582554]
Is sorted (descending): True
✓ Test 4 passed

Test 5: Verify data centering
Original data mean: [10.09176598  9.81676669 10.07482166]
Stored mean: [10.09176598  9.81676669 10.07482166]
Difference: 0.0000000000
✓ Test 5 passed

ALL TESTS PASSED!


## PCA on Real-World Data

* Applying the PCA implementation on the `California Housing Dataset`.
* Comparing the results with those obtained from the scikit-learn PCA implementation: `sklearn.decomposition.PCA`.


In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.decomposition import PCA as SklearnPCA
from sklearn.preprocessing import StandardScaler


In [ ]:
# Load data
housing = fetch_california_housing()
X = housing.data
y = housing.target

# Standardize data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Custom PCA (with updated fit)
custom_pca = PCA(n_components=5)
X_custom_transformed = custom_pca.fit_transform(X_scaled)

# Scikit-learn PCA
sklearn_pca = SklearnPCA(n_components=5)
X_sklearn_transformed = sklearn_pca.fit_transform(X_scaled)

# Align custom components to sklearn
for i in range(custom_pca.n_components):
    if np.dot(custom_pca.components_[i], sklearn_pca.components_[i]) < 0:
        custom_pca.components_[i] *= -1
        X_custom_transformed[:, i] *= -1  # Flip transformed coordinates too

# Now compare transformed data
diff_transformed = np.mean(np.abs(X_custom_transformed - X_sklearn_transformed))
print(f"Mean absolute difference in transformed data (after alignment): {diff_transformed}")

# Compare explained variance
diff_variance = np.mean(np.abs(custom_pca.explained_variance_ - sklearn_pca.explained_variance_))
print(f"Mean absolute difference in explained variance: {diff_variance}")

# Assert numerical equivalence
assert diff_transformed < 1e-10, f"Transformed data not equivalent: {diff_transformed}"
assert diff_variance < 1e-10, f"Explained variance not equivalent: {diff_variance}"

print("Custom PCA matches scikit-learn within numerical precision!")


Mean absolute difference in transformed data (after alignment): 1.6208898210626536e-15
Mean absolute difference in explained variance: 4.440892098500626e-16
Custom PCA matches scikit-learn within numerical precision!
